In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(r"C:\temp\shift_ml\train_data")
PROCESSED_DIR = DATA_DIR.parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_test = pd.read_csv(DATA_DIR / "X_test_base.csv")
y_train = pd.read_csv(DATA_DIR / "y_train_base.csv")

X_test["car_id"] = X_test["car_id"].astype("string").str.strip()
y_train["car_id"] = y_train["car_id"].astype("string").str.strip()


def read_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix == ".xlsx":
        return pd.read_excel(path)

    if suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)

    raise ValueError(f"Неподдерживаемый формат: {suffix}")


def dataset_number(path: Path) -> int:
    return int(path.parent.name.split("_")[-1])


dataset_files = sorted(
    DATA_DIR.glob("dataset_*/data.*"),
    key=dataset_number
)

raw_parts = []

for path in dataset_files:
    df = read_dataset(path).copy()
    df["car_id"] = df["car_id"].astype("string").str.strip()
    raw_parts.append(df)

train_raw = pd.concat(raw_parts, ignore_index=True)

print("Сырые train-данные:", train_raw.shape)
print("Уникальных car_id:", train_raw["car_id"].nunique())

Сырые train-данные: (49679, 22)
Уникальных car_id: 8340


Теперь агрегируем строки. Здесь для каждого столбца берём первое найденное непустое значение. Это безопасно именно потому, что ранее мы проверили отсутствие конфликтов.

In [2]:
feature_columns = [column for column in X_test.columns if column != "car_id"]


def first_non_null(series: pd.Series):
    non_null_values = series.dropna()

    if non_null_values.empty:
        return pd.NA

    return non_null_values.iloc[0]


X_train = (
    train_raw
    .groupby("car_id", sort=False)[feature_columns]
    .agg(first_non_null)
    .reset_index()
)

# Ставим порядок колонок точно как в X_test.
X_train = X_train[X_test.columns]

print("X_train после агрегации:", X_train.shape)
print("X_test:", X_test.shape)

assert X_train.shape[0] == 8340
assert X_train["car_id"].nunique() == 8340
assert list(X_train.columns) == list(X_test.columns)
assert X_train["car_id"].duplicated().sum() == 0

X_train после агрегации: (8340, 22)
X_test: (8341, 22)


Соединяем признаки и target только по идентификатору:

In [3]:
missing_tokens = {
    "",
    "-",
    "null",
    "none",
    "nan",
    "n/a",
}

def clean_offer_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    offer = (
        df["Предложение"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace(missing_tokens, pd.NA)
    )

    df["Предложение"] = pd.to_numeric(
        offer,
        errors="coerce",
    )

    return df


X_train = clean_offer_column(X_train)
X_test = clean_offer_column(X_test)

In [4]:
train = X_train.merge(
    y_train,
    on="car_id",
    how="inner",
    validate="one_to_one",
)

print("Итоговый train:", train.shape)

assert len(train) == len(y_train)
assert train["Цена"].isna().sum() == 0
assert train["car_id"].nunique() == len(train)

display(train.head())

Итоговый train: (8340, 23)


,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,...,Цвет,Локация,Количество цилиндров,Тип кузова,Двери,Количество кресел,Оценка эксперта,Количество владельцев,Предложение,Цена
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,2022.0,HAVAL,SPRINGWOOD GWM HAVAL,2022 GWM HAVAL H6 ULTRA AWD,DEMO,Automatic,"4 cyl, 2 L",AWD,...,Grey / Black,"SPRINGWOOD, QLD",4 cyl,SUV,4 Doors,5 Seats,4.0,5.0,32145.0,38812
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,2014.0,TERRITORY,SUV,2014 FORD TERRITORY TITANIUM (4X4),USED,Automatic,"6 cyl, 2.7 L",AWD,...,White / -,"PENRITH, NSW",6 cyl,SUV,4 Doors,7 Seats,6.0,6.0,15949.0,15950
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,2021.0,X-TRAIL,SUV,2021 NISSAN X-TRAIL TI (4WD),USED,Automatic,"4 cyl, 2.5 L",4WD,...,White / -,"WEST FOOTSCRAY, VIC",4 cyl,SUV,4 Doors,5 Seats,4.0,7.0,39616.0,41990
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,2022.0,XC60,SUV,2022 VOLVO XC60 B5 MOMENTUM MHEV,USED,Automatic,"4 cyl, 2 L",AWD,...,White / Black,"SOUTH BUNBURY, WA",4 cyl,SUV,4 Doors,5 Seats,1.0,9.0,70449.0,69900
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,2018.0,KOLEOS,SUV,2018 RENAULT KOLEOS INTENS (4X4),USED,Automatic,"4 cyl, 2 L",4WD,...,White / -,"RINGWOOD, VIC",4 cyl,SUV,4 Doors,5 Seats,1.0,8.0,22895.0,27950


Сравниваем пропуски train и test после агрегации:

In [5]:
missingness_report = pd.DataFrame(
    {
        "train_missing_pct": X_train[feature_columns].isna().mean().mul(100),
        "test_missing_pct": X_test[feature_columns].isna().mean().mul(100),
    }
)

missingness_report["difference_test_minus_train"] = (
    missingness_report["test_missing_pct"]
    - missingness_report["train_missing_pct"]
)

missingness_report = (
    missingness_report
    .sort_values("test_missing_pct", ascending=False)
    .round(2)
)

display(missingness_report)

,train_missing_pct,test_missing_pct,difference_test_minus_train
Предложение,0.00,100.00,100.00
Количество кресел,9.76,10.48,0.72
Двери,9.20,9.83,0.63
Локация,2.55,2.83,0.28
Тип кузова,1.59,1.75,0.16
Тип машины,0.16,0.16,-0.00
Модель,0.00,0.00,0.00
Год выпуска,0.00,0.00,0.00
Бренд,0.00,0.00,0.00
Полное название,0.00,0.00,0.00


Проверяем столбец Предложение:

In [6]:
offer = pd.to_numeric(train["Предложение"], errors="coerce")
price = train["Цена"]

mask = offer.notna() & price.notna()

offer_audit = pd.DataFrame(
    {
        "metric": [
            "Всего объектов",
            "Непустых значений в Предложение",
            "Доля непустых Предложение, %",
            "Точных совпадений Предложение и Цена",
            "Доля точных совпадений среди непустых, %",
            "Корреляция Предложение и Цена",
            "MAPE, если прогнозировать значением Предложение, %",
        ],
        "value": [
            len(train),
            int(mask.sum()),
            round(mask.mean() * 100, 2),
            int((offer[mask] == price[mask]).sum()),
            round((offer[mask] == price[mask]).mean() * 100, 2),
            round(offer[mask].corr(price[mask]), 4),
            round(
                (
                    np.abs(price[mask] - offer[mask])
                    / price[mask]
                ).mean() * 100,
                4,
            ),
        ],
    }
)

display(offer_audit)

display(
    train.loc[
        mask,
        ["car_id", "Бренд", "Модель", "Год выпуска", "Предложение", "Цена"]
    ].head(10)
)

,metric,value
0,Всего объектов,8340.0000
1,Непустых значений в Предложение,8340.0000
2,"Доля непустых Предложение, %",100.0000
3,Точных совпадений Предложение и Цена,0.0000
4,"Доля точных совпадений среди непустых, %",0.0000
5,Корреляция Предложение и Цена,0.9938
6,"MAPE, если прогнозировать значением Предложени...",8.3118


,car_id,Бренд,Модель,Год выпуска,Предложение,Цена
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,HAVAL,2022.0,32145.0,38812
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,TERRITORY,2014.0,15949.0,15950
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,X-TRAIL,2021.0,39616.0,41990
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,XC60,2022.0,70449.0,69900
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,KOLEOS,2018.0,22895.0,27950
5,85356769-9239-48d6-9c83-a918b7a8e66f,KIA,PICANTO,2019.0,20659.0,20999
6,5e974caa-66e1-4be3-8899-67dd5c116bb0,AUDI,A1,2021.0,27832.0,32888
7,97c4619a-0a2b-4f26-aa57-d5ccd780bd79,MITSUBISHI,TRITON,2022.0,50290.0,55990
8,f9a39f6d-cfdb-475a-a2b2-d4a3d414cffe,TOYOTA,RAV4,2019.0,35064.0,36990
9,c322cca0-b4f2-42be-b40a-df689287c6e0,FORD,FAIRLANE,2004.0,7466.0,8999


И сохраняем собранные данные как промежуточный воспроизводимый артефакт:

In [7]:
X_train.to_parquet(
    PROCESSED_DIR / "X_train_canonical.parquet",
    index=False,
)

train.to_parquet(
    PROCESSED_DIR / "train_canonical.parquet",
    index=False,
)

X_test.to_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet",
    index=False,
)

Нельзя оставлять Предложение даже с SimpleImputer: на validation он будет полезен, а на test будет заполнен одной константой.

Нельзя использовать его для feature engineering, target encoding или выбора моделей.

Нельзя оценивать модель на split, где Предложение доступно и в train-, и в validation-части.

Его можно оставить только как предмет EDA: например, зафиксировать в отчёте, что признак исключён из-за полной недоступности в test и почти прямой связи с target.

Это самый важный leakage-риск, который мы пока обнаружили.

In [8]:
ID_COLUMN = "car_id"
TARGET_COLUMN = "Цена"

EXCLUDED_COLUMNS = [
    ID_COLUMN,
    TARGET_COLUMN,
    "Предложение",
]

feature_columns = [
    col for col in train.columns
    if col not in EXCLUDED_COLUMNS
]

print(f"Количество признаков: {len(feature_columns)}")
print(feature_columns)

assert "Предложение" not in feature_columns
assert ID_COLUMN not in feature_columns
assert TARGET_COLUMN not in feature_columns

X = train[feature_columns].copy()
y = train[TARGET_COLUMN].copy()

X_final_test = X_test[feature_columns].copy()

assert list(X.columns) == list(X_final_test.columns)
assert len(X) == 8340
assert len(X_final_test) == 8341

Количество признаков: 20
['Бренд', 'Год выпуска', 'Модель', 'Тип машины', 'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод', 'Топливо', 'Расход', 'Пробег', 'Цвет', 'Локация', 'Количество цилиндров', 'Тип кузова', 'Двери', 'Количество кресел', 'Оценка эксперта', 'Количество владельцев']
